<a href="https://colab.research.google.com/github/JakeOh/202605_BD57/blob/main/lab_ml/ml04_cancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

유방암 데이터 셋에서 양성/음성 분류

# Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets  # 예제 데이터 셋 모듈. load_xyz() 함수들.
from sklearn.model_selection import train_test_split  # 훈련 셋 vs 테스트 셋 분리
from sklearn.preprocessing import StandardScaler  # 표준화 특성 스케일러
from sklearn.neighbors import KNeighborsClassifier  # KNN 분류 모델
from sklearn.metrics import classification_report, confusion_matrix  # 평가 지표

# Breast Cancer 데이터 셋

In [2]:
# scikit-learn 패키지의 예제 데이터 셋 가져오기
cancer = datasets.load_breast_cancer()

In [4]:
print(type(cancer))

<class 'sklearn.utils._bunch.Bunch'>


scikit-learn 패키지의 `Bunch` 클래스:

*   파이썬의 `dict`와 비슷한 클래스
*   key-value 쌍으로 아이템들을 저장.
*   `bunch_name['key']` 또는 `bunch_name.key` 형식으로 키(key)를 사용해서 값(value)을 참조할 수 있음.

In [5]:
cancer.keys()  # Bunch 객체의 키들.

dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])

In [7]:
# 데이터 셋 설명(description)
print(cancer.DESCR)  # print(cancer['DESCR'])

.. _breast_cancer_dataset:

Breast cancer wisconsin (diagnostic) dataset
--------------------------------------------

**Data Set Characteristics:**

:Number of Instances: 569

:Number of Attributes: 30 numeric, predictive attributes and the class

:Attribute Information:
    - radius (mean of distances from center to points on the perimeter)
    - texture (standard deviation of gray-scale values)
    - perimeter
    - area
    - smoothness (local variation in radius lengths)
    - compactness (perimeter^2 / area - 1.0)
    - concavity (severity of concave portions of the contour)
    - concave points (number of concave portions of the contour)
    - symmetry
    - fractal dimension ("coastline approximation" - 1)

    The mean, standard error, and "worst" or largest (mean of the three
    worst/largest values) of these features were computed for each image,
    resulting in 30 features.  For instance, field 0 is Mean Radius, field
    10 is Radius SE, field 20 is Worst Radius.

    - 

In [8]:
print(cancer.target_names)
#> 0 -> malignant(양성, 악성 암), 1 -> benign(음성, 암이 아님.)

['malignant' 'benign']


In [9]:
print(cancer.feature_names)  #> 특성 이름들. 데이터프레임의 컬럼 이름들.

['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']


In [10]:
x = cancer.data  # 특성 배열(2차원 배열)
y = cancer.target  # 타겟 배열(1차원 배열)

In [11]:
x.shape  # (n_samples, n_features) = (539, 30): 539개 진단 데이터, 30개의 변수(컬럼).

(569, 30)

In [12]:
y.shape  # (n_sample,) = (539,): 539개 데이터의 레이블들

(569,)

In [13]:
x[:2]

array([[1.799e+01, 1.038e+01, 1.228e+02, 1.001e+03, 1.184e-01, 2.776e-01,
        3.001e-01, 1.471e-01, 2.419e-01, 7.871e-02, 1.095e+00, 9.053e-01,
        8.589e+00, 1.534e+02, 6.399e-03, 4.904e-02, 5.373e-02, 1.587e-02,
        3.003e-02, 6.193e-03, 2.538e+01, 1.733e+01, 1.846e+02, 2.019e+03,
        1.622e-01, 6.656e-01, 7.119e-01, 2.654e-01, 4.601e-01, 1.189e-01],
       [2.057e+01, 1.777e+01, 1.329e+02, 1.326e+03, 8.474e-02, 7.864e-02,
        8.690e-02, 7.017e-02, 1.812e-01, 5.667e-02, 5.435e-01, 7.339e-01,
        3.398e+00, 7.408e+01, 5.225e-03, 1.308e-02, 1.860e-02, 1.340e-02,
        1.389e-02, 3.532e-03, 2.499e+01, 2.341e+01, 1.588e+02, 1.956e+03,
        1.238e-01, 1.866e-01, 2.416e-01, 1.860e-01, 2.750e-01, 8.902e-02]])

In [14]:
y[:2]

array([0, 0])

In [15]:
y[-2:]

array([0, 1])

In [16]:
# 0(malignant) vs 1(benign)의 비율
pd.Series(y).value_counts()

,count
1,357
0,212
